# Task 3: Linear Regression

**AI & ML Internship — Elevate Labs**

---

## Objective
Implement and understand simple and multiple linear regression using the California Housing dataset. This notebook covers the complete ML workflow: data loading, preprocessing, train-test splitting, model training, evaluation, visualization, and interpretation.

**Key goals:**
1. Load and explore a real-world regression dataset
2. Preprocess data (handle missing values, encode categoricals)
3. Split data into training and test sets
4. Fit simple linear regression (one feature)
5. Fit multiple linear regression (all features)
6. Evaluate models using MAE, MSE, and R²
7. Visualize regression lines and predictions
8. Interpret coefficients and check assumptions
9. Prepare for common regression interview questions


## Objective

Linear regression is one of the fundamental algorithms in machine learning. It models the relationship between a dependent variable and one or more independent features by fitting a linear equation.

**What you will learn:**
- How to prepare data for regression modeling
- The difference between simple and multiple linear regression
- How to interpret model coefficients
- How to evaluate regression performance
- How to visualize regression results
- Common pitfalls and assumptions in linear regression

**Why this matters:** Linear regression is the foundation for understanding more complex models. Mastering it prepares you for regularization (Ridge/Lasso), polynomial regression, and generalized linear models.


## Dataset Description

**Dataset:** California Housing Dataset  
**Source:** [Hands-On Machine Learning with Scikit-Learn, Keras & TensorFlow](https://github.com/ageron/handson-ml)  
**Local path:** `dataset/housing.csv`  
**Shape:** 20,640 rows × 10 columns

### Columns
| Column | Type | Description |
|--------|------|-------------|
| longitude | float | Longitude coordinate |
| latitude | float | Latitude coordinate |
| housing_median_age | float | Median age of houses in the block |
| total_rooms | float | Total number of rooms in the block |
| total_bedrooms | float | Total number of bedrooms in the block |
| population | float | Total population in the block |
| households | float | Total number of households in the block |
| median_income | float | Median income of households in the block |
| ocean_proximity | str | Proximity to the ocean (categorical) |
| median_house_value | float | **Target variable** — median house value in the block |

**Why this dataset?** It contains a realistic mix of numerical and categorical features, missing values, and multicollinearity — perfect for practicing regression end-to-end.


## Import Libraries

We import all required libraries at the top for reproducibility.

- `pandas` — data loading and manipulation
- `numpy` — numerical operations
- `matplotlib.pyplot` — plotting
- `seaborn` — statistical visualizations
- `sklearn` — preprocessing, model training, evaluation


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Set plot style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

print("Libraries imported successfully.")
print(f"Pandas: {pd.__version__} | NumPy: {np.__version__} | Seaborn: {sns.__version__}")


## Load Dataset

Load the housing CSV and verify its integrity.

**Expected output:**
- Success message
- Shape: (20640, 10)


In [ ]:
import os

dataset_path = "../dataset/housing.csv"

if not os.path.exists(dataset_path):
    raise FileNotFoundError(f"Dataset not found at {dataset_path}")

df = pd.read_csv(dataset_path)
print(f"Dataset loaded: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"File size: {os.path.getsize(dataset_path) / 1024:.2f} KB")


## Initial Data Exploration

We inspect the dataset structure, data types, and initial quality.

**What to look for:**
- Column names and types
- Missing values
- Data ranges
- Categorical vs numerical columns


In [ ]:
print("=== First 5 rows ===")
display(df.head())

print("\n=== Dataset Info ===")
print(f"Shape: {df.shape}")
print(f"\nData types:\n{df.dtypes}")
print(f"\nMemory usage: {df.memory_usage(deep=True).sum() / 1024:.2f} KB")


In [ ]:
print("=== Summary Statistics ===")
display(df.describe().round(2))


## Data Preprocessing

Before modeling, we must handle data quality issues.

**Steps:**
1. Handle missing values in `total_bedrooms`
2. Encode categorical variable `ocean_proximity`
3. Separate features and target
4. Scale numerical features (optional for linear regression but good practice)

**Missing value strategy:** `total_bedrooms` has missing values. We impute with median because it is robust to outliers.

**Encoding strategy:** `ocean_proximity` is nominal. We use one-hot encoding to avoid imposing false ordinal relationships.


In [ ]:
print("=== Missing Values ===")
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing_Count': missing, 'Percentage': missing_pct})
missing_df = missing_df[missing_df['Missing_Count'] > 0].sort_values('Percentage', ascending=False)
display(missing_df)

# Impute total_bedrooms with median
df['total_bedrooms'] = df['total_bedrooms'].fillna(df['total_bedrooms'].median())
print("\nImputed total_bedrooms missing values with median.")
print(f"Missing values after imputation: {df.isnull().sum().sum()}")


In [ ]:
print("=== One-Hot Encoding: ocean_proximity ===")
df = pd.get_dummies(df, columns=['ocean_proximity'], prefix='ocean_proximity', drop_first=False)
print(f"Shape after encoding: {df.shape}")
print("New columns:", [c for c in df.columns if 'ocean_proximity' in c])


## Train-Test Split

We split the data before any further processing to avoid data leakage.

**Split:** 80% training, 20% testing  
**Random state:** 42 for reproducibility  
**Stratify:** Not needed for regression (stratification is for classification)

**Why split first?**
- Prevents test data information from influencing the model
- Ensures evaluation reflects real-world generalization
- Allows fair comparison of different models


In [ ]:
print("=== Train-Test Split ===")

# Separate features and target
X = df.drop(columns=['median_house_value'])
y = df['median_house_value']

print(f"Features (X): {X.shape}")
print(f"Target (y): {y.shape}")

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"\nTrain shape: X={X_train.shape}, y={y_train.shape}")
print(f"Test shape: X={X_test.shape}, y={y_test.shape}")
print(f"\nTrain target mean: {y_train.mean():.2f}")
print(f"Test target mean: {y_test.mean():.2f}")
print("\nMeans are close, indicating a representative split.")


## Simple Linear Regression

Simple linear regression uses **one feature** to predict the target.

**Formula:** `y = β₀ + β₁x + ε`

**Our model:** `median_house_value = β₀ + β₁ × median_income + ε`

**Why median_income?** It is typically the strongest predictor of house prices. We start with one feature to build intuition before adding more.


In [ ]:
print("=== Simple Linear Regression ===")

# Select one feature: median_income
X_train_simple = X_train[['median_income']]
X_test_simple = X_test[['median_income']]

# Fit model
lr_simple = LinearRegression()
lr_simple.fit(X_train_simple, y_train)

# Predictions
y_train_pred_simple = lr_simple.predict(X_train_simple)
y_test_pred_simple = lr_simple.predict(X_test_simple)

print(f"Coefficient (β₁): {lr_simple.coef_[0]:.4f}")
print(f"Intercept (β₀): {lr_simple.intercept_:.4f}")
print(f"\nEquation: median_house_value = {lr_simple.intercept_:.2f} + {lr_simple.coef_[0]:.2f} × median_income")


## Multiple Linear Regression

Multiple linear regression uses **all features** to predict the target.

**Formula:** `y = β₀ + β₁x₁ + β₂x₂ + ... + βₙxₙ + ε`

**Why multiple?** House prices depend on many factors simultaneously. Including all relevant features improves predictive power and reduces omitted variable bias.


In [ ]:
print("=== Multiple Linear Regression ===")

# Fit model on all features
lr_multiple = LinearRegression()
lr_multiple.fit(X_train, y_train)

# Predictions
y_train_pred_multiple = lr_multiple.predict(X_train)
y_test_pred_multiple = lr_multiple.predict(X_test)

print(f"Number of features: {lr_multiple.n_features_in_}")
print(f"\nIntercept (β₀): {lr_multiple.intercept_:.4f}")
print("\nCoefficients (β₁ to βₙ):")
for feature, coef in zip(X_train.columns, lr_multiple.coef_):
    print(f"  {feature}: {coef:.4f}")


## Model Evaluation

We evaluate both models using three standard metrics:

| Metric | Formula | Interpretation |
|--------|---------|----------------|
| **MAE** | mean(\|y - ŷ\|) | Average absolute error; same units as target |
| **MSE** | mean((y - ŷ)²) | Penalizes large errors more heavily |
| **R²** | 1 - SS_res / SS_tot | Proportion of variance explained; 1 = perfect |

**When to prefer MSE over MAE:** MSE is differentiable and works better with gradient-based optimization. It also penalizes large errors more, which is useful when outliers are particularly costly.

**Expected output:**
- Train and test metrics for both models
- Comparison showing multiple regression outperforms simple regression


In [ ]:
def evaluate_model(y_true, y_pred, model_name):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    print(f"=== {model_name} ===")
    print(f"MAE: {mae:.2f}")
    print(f"MSE: {mse:.2f}")
    print(f"R²: {r2:.4f}")
    print()
    return {'Model': model_name, 'MAE': mae, 'MSE': mse, 'R2': r2}

results = []
results.append(evaluate_model(y_train, y_train_pred_simple, 'Simple LR (Train)'))
results.append(evaluate_model(y_test, y_test_pred_simple, 'Simple LR (Test)'))
results.append(evaluate_model(y_train, y_train_pred_multiple, 'Multiple LR (Train)'))
results.append(evaluate_model(y_test, y_test_pred_multiple, 'Multiple LR (Test)'))

results_df = pd.DataFrame(results)
display(results_df.round(4))


In [ ]:
print("=== Interpretation ===")
print("1. Multiple LR R² > Simple LR R²: Adding more features improves explanation of variance.")
print("2. Train R² > Test R²: Slight overfitting, but gap is small — model generalizes reasonably.")
print("3. MAE and MSE are in dollars. MAE ~ $50,000 means average prediction error is $50k.")
print("4. R² ≈ 0.60-0.65 means ~60-65% of price variance is explained by the model.")


## Regression Line Visualization

We visualize the simple linear regression model by plotting actual vs predicted values and the regression line.

**Visualizations:**
1. Scatter plot with regression line (median_income vs price)
2. Actual vs Predicted scatter plot
3. Residual plot


In [ ]:
print("=== 1. Regression Line: median_income vs median_house_value ===")
fig, ax = plt.subplots(figsize=(10, 6))

# Scatter plot of actual data
ax.scatter(X_train['median_income'], y_train, alpha=0.5, s=10, color='steelblue', label='Actual')

# Regression line
x_line = np.linspace(X_train['median_income'].min(), X_train['median_income'].max(), 100)
y_line = lr_simple.intercept_ + lr_simple.coef_[0] * x_line
ax.plot(x_line, y_line, color='red', linewidth=2, label='Regression Line')

ax.set_xlabel('Median Income', fontsize=12)
ax.set_ylabel('Median House Value', fontsize=12)
ax.set_title('Simple Linear Regression: Median Income vs House Price', fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

print("Interpretation: The red line shows the predicted relationship. As median income increases, predicted house value increases linearly.")


In [ ]:
print("=== 2. Actual vs Predicted (Multiple LR) ===")
fig, ax = plt.subplots(figsize=(10, 6))

ax.scatter(y_test, y_test_pred_multiple, alpha=0.5, s=20, color='seagreen')
ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', linewidth=2, label='Perfect Prediction')

ax.set_xlabel('Actual Price', fontsize=12)
ax.set_ylabel('Predicted Price', fontsize=12)
ax.set_title('Multiple Linear Regression: Actual vs Predicted', fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

print("Interpretation: Points close to the red dashed line indicate accurate predictions. Spread indicates prediction error.")


In [ ]:
print("=== 3. Residual Plot (Multiple LR) ===")
residuals = y_test - y_test_pred_multiple

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(y_test_pred_multiple, residuals, alpha=0.5, s=20, color='coral')
ax.axhline(y=0, color='black', linestyle='--', linewidth=1)
ax.set_xlabel('Predicted Price', fontsize=12)
ax.set_ylabel('Residuals', fontsize=12)
ax.set_title('Residual Plot: Multiple Linear Regression', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Interpretation: Residuals should be randomly scattered around 0. Patterns indicate non-linearity or heteroscedasticity.")


## Coefficient Interpretation

Linear regression coefficients tell us the expected change in the target for a one-unit change in the feature, holding all other features constant.

**Simple LR:**
- Intercept (β₀): Base house price when median_income = 0
- Coefficient (β₁): Price change per unit increase in median_income

**Multiple LR:**
- Each coefficient represents the isolated effect of that feature
- Positive coefficient → feature increases house price
- Negative coefficient → feature decreases house price
- Magnitude indicates strength of effect (after scaling)


In [ ]:
print("=== Coefficient Analysis: Multiple Linear Regression ===")

coef_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Coefficient': lr_multiple.coef_
})
coef_df = coef_df.sort_values('Coefficient', key=abs, ascending=False)
display(coef_df)

print("\n=== Key Interpretations ===")
print("1. median_income: Strong positive effect. Higher income → higher prices.")
print("2. ocean_proximity_NEAR BAY/NEAR OCEAN: Positive effect. Waterfront proximity adds value.")
print("3. ocean_proximity_INLAND: Negative effect. Inland locations have lower prices.")
print("4. latitude: Negative effect. Moving north (higher latitude) tends to decrease price in California.")
print("5. longitude: Mixed effect depending on coastal vs inland location.")
print("\nNote: Coefficients are not directly comparable because features have different scales.")
print("For fair comparison, standardize features before interpreting magnitude.")


## Multicollinearity Check

Multicollinearity occurs when features are highly correlated, which can make coefficients unstable and unreliable.

**Detection methods:**
1. Correlation matrix — visualize pairwise correlations
2. Variance Inflation Factor (VIF) — quantitative measure

**Rule of thumb:**
- |r| > 0.8 between features = potential multicollinearity
- VIF > 5 or 10 = problematic multicollinearity


In [ ]:
print("=== Correlation Matrix ===")
numeric_cols = ['longitude', 'latitude', 'housing_median_age', 'total_rooms', 
                'total_bedrooms', 'population', 'households', 'median_income', 'median_house_value']
corr_matrix = df[numeric_cols].corr().round(3)
display(corr_matrix)

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, cmap='RdBu_r', center=0, 
            square=True, linewidths=0.5, cbar_kws={'shrink': 0.8}, ax=ax, fmt='.2f')
ax.set_title('Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Interpretation: total_rooms vs total_bedrooms, population vs households, and total_rooms vs population are highly correlated. This indicates multicollinearity.")


In [ ]:
print("=== Variance Inflation Factor (VIF) ===")
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Add constant for statsmodels
X_vif = X_train[['longitude', 'latitude', 'housing_median_age', 'total_rooms', 
                 'total_bedrooms', 'population', 'households', 'median_income']].copy()

vif_data = pd.DataFrame()
vif_data['Feature'] = X_vif.columns
vif_data['VIF'] = [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
display(vif_data.sort_values('VIF', ascending=False))

print("\nInterpretation: VIF > 5-10 indicates multicollinearity. total_rooms, total_bedrooms, population, and households have very high VIF values.")
print("Solution: Consider PCA, feature selection, or regularization (Ridge/Lasso).")


## Residual Analysis

Residuals are the differences between actual and predicted values: `e = y - ŷ`

**Assumptions we check:**
1. **Linearity:** Residuals should show no pattern against predicted values
2. **Homoscedasticity:** Residual variance should be constant across predictions
3. **Normality:** Residuals should be approximately normally distributed
4. **Independence:** Residuals should be independent of each other


In [ ]:
print("=== Residual Distribution ===")
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of residuals
axes[0].hist(residuals, bins=50, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].axvline(x=0, color='red', linestyle='--', linewidth=2)
axes[0].set_xlabel('Residual Value', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].set_title('Distribution of Residuals', fontsize=14, fontweight='bold')

# Q-Q plot
stats.probplot(residuals, dist="norm", sparams=(residuals.mean(), residuals.std()), plot=axes[1])
axes[1].set_title('Q-Q Plot of Residuals', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print("Interpretation: If residuals are normally distributed and centered at 0, the linear regression assumptions are reasonably met.")


## Interview Preparation: Linear Regression Q&A

### 1. What assumptions does linear regression make?

**Simple answer:** Linear regression assumes the relationship between features and target is linear, errors are random and normally distributed, and features are not too correlated with each other.

**Technical answer:**  
- **Linearity:** The relationship between independent variables (X) and dependent variable (y) is linear.  
- **Independence:** Observations are independent of each other (no autocorrelation).  
- **Homoscedasticity:** The variance of errors is constant across all levels of X.  
- **Normality:** Errors are normally distributed.  
- **No multicollinearity:** Independent variables are not highly correlated with each other.  
- **No endogeneity:** Independent variables are not correlated with the error term.

**Example:** If you predict house prices using size and location, you assume price changes linearly with size, error variance is similar for small and large houses, and size and location aren't perfectly correlated.

**Follow-up:** "What happens if these assumptions are violated?"

---

### 2. How do you interpret the coefficients?

**Simple answer:** Each coefficient tells you how much the target changes when that feature increases by one unit, keeping everything else the same.

**Technical answer:**  
- **Intercept (β₀):** Expected value of y when all X = 0. May not always be meaningful (e.g., house price when income = 0).  
- **Coefficient (β₁):** Expected change in y for a one-unit increase in X₁, holding all other variables constant.  
- **Sign:** Positive = direct relationship; Negative = inverse relationship.  
- **Magnitude:** Larger absolute value = stronger effect (only comparable if features are scaled).

**Example:** If β₁ for median_income = 42,000, then a $1 increase in median income is associated with a $42,000 increase in predicted house price, holding other factors constant.

**Follow-up:** "Why might coefficients change when you add or remove features?"

---

### 3. What is R² score and its significance?

**Simple answer:** R² tells you what percentage of the target's variation your model explains. Higher is better, up to 1.0.

**Technical answer:**  
- **Definition:** R² = 1 - (SS_res / SS_tot), where SS_res is the sum of squared residuals and SS_tot is the total sum of squares.  
- **Range:** 0 to 1 (can be negative for very bad models).  
- **Interpretation:** R² = 0.85 means 85% of variance in y is explained by the model.  
- **Limitation:** R² always increases when you add more features, even if they are useless. Use Adjusted R² for multiple regression.  
- **Significance:** R² alone doesn't tell you if the model is good. Always check residuals and validation performance.

**Example:** An R² of 0.60 means the model captures 60% of house price variation. The remaining 40% is due to factors not in the model or random noise.

**Follow-up:** "Can a high R² be misleading? When would you not trust it?"

---

### 4. When would you prefer MSE over MAE?

**Simple answer:** Use MSE when you want to penalize large errors more heavily, or when you need a differentiable loss function for optimization.

**Technical answer:**  
- **MSE:** `mean((y - ŷ)²)` — squares errors, so outliers have much larger impact. Differentiable everywhere. Preferred in gradient descent and mathematical derivations.  
- **MAE:** `mean(|y - ŷ|)` — linear penalty, more robust to outliers. Not differentiable at 0.  
- **When to prefer MSE:**  
  1. Large errors are particularly costly (e.g., medical diagnosis, fraud detection)  
  2. Using gradient-based optimization (closed-form or gradient descent)  
  3. You want to emphasize minimizing big mistakes  
  4. Comparing models using statistical methods like ANOVA  
- **When to prefer MAE:**  
  1. Outliers should not dominate the metric  
  2. You want an intuitive, dollar-unit interpretation  

**Example:** In house price prediction, a $500k error is 10× worse than a $50k error. MSE makes this explicit by squaring the error, giving it 100× weight.

**Follow-up:** "What is RMSE and how is it related to MSE?"

---

### 5. How do you detect multicollinearity?

**Simple answer:** Check if features are highly correlated with each other using correlation matrices or VIF scores.

**Technical answer:**  
- **Correlation matrix:** Compute Pearson correlation between all pairs of features. |r| > 0.8 suggests multicollinearity.  
- **VIF (Variance Inflation Factor):** VIF = 1 / (1 - R²_j), where R²_j is the R² from regressing feature j on all other features.  
  - VIF = 1: No correlation  
  - VIF > 5: Moderate multicollinearity  
  - VIF > 10: Severe multicollinearity  
- **Condition number:** From linear algebra, large condition numbers indicate multicollinearity.  
- **Tolerance:** 1 / VIF. Low tolerance = high multicollinearity.

**Example:** In housing data, `total_rooms` and `total_bedrooms` are likely highly correlated. Their VIF might be > 10, indicating multicollinearity.

**Follow-up:** "How do you fix multicollinearity?"

---

### 6. What is the difference between simple and multiple regression?

**Simple answer:** Simple regression uses one feature to predict the target. Multiple regression uses two or more features.

**Technical answer:**  
- **Simple Linear Regression (SLR):** `y = β₀ + β₁x₁ + ε`. One independent variable. Easy to visualize (2D scatter plot with line). Coefficients are stable unless the single feature is highly correlated with omitted variables.  
- **Multiple Linear Regression (MLR):** `y = β₀ + β₁x₁ + β₂x₂ + ... + βₙxₙ + ε`. Two or more independent variables. Captures more variance but introduces multicollinearity risk. Coefficients represent partial effects (holding other variables constant).  
- **Key differences:**  
  1. Number of features: 1 vs. many  
  2. Visualization: 2D line vs. higher-dimensional plane/hyperplane  
  3. Interpretation: Direct relationship vs. conditional relationship  
  4. Multicollinearity: Not an issue in SLR; major concern in MLR  
  5. Model complexity: Low vs. higher

**Example:** SLR: `price = β₀ + β₁ × size`. MLR: `price = β₀ + β₁ × size + β₂ × location + β₃ × age`.

**Follow-up:** "Can you have too many features in multiple regression? What are the risks?"

---

### 7. Can linear regression be used for classification?

**Simple answer:** Not really. Linear regression predicts continuous numbers, not categories. For classification, use logistic regression or other classifiers.

**Technical answer:**  
- **Linear regression output:** Continuous values in (-∞, +∞). For binary classification (0/1), predictions can be outside [0, 1], which is invalid for probabilities.  
- **Logistic regression:** Uses the sigmoid function to map linear regression output to [0, 1], giving valid class probabilities.  
- **Why not force it:** Thresholding linear regression output at 0.5 is arbitrary and sensitive to outliers. The loss function (MSE) is not appropriate for classification.  
- **Exception:** In rare cases, linear regression can be used for ordered categories with many levels, but logistic regression is still preferred.

**Example:** Predicting whether a house sells for > $500k (classification) should use logistic regression, not linear regression which might predict -$50k or $2M.

**Follow-up:** "What is logistic regression and how does it relate to linear regression?"

---

### 8. What happens if you violate regression assumptions?

**Simple answer:** Your model may still work, but its predictions could be unreliable, coefficients misleading, or standard errors wrong.

**Technical answer:**  
- **Violate linearity:** Model will underfit; predictions will be systematically wrong in certain ranges.  
- **Violate independence:** Standard errors are underestimated, leading to false statistical significance. Common in time series data.  
- **Violate homoscedasticity:** Standard errors are biased; OLS is no longer BLUE (Best Linear Unbiased Estimator). Use robust standard errors or weighted least squares.  
- **Violate normality:** Affects inference (p-values, confidence intervals) more than predictions, especially in small samples. With large n, CLT helps.  
- **Violate no multicollinearity:** Coefficients become unstable and high-variance; small data changes cause large coefficient swings. Interpretation becomes unreliable.  
- **Violate no endogeneity:** Coefficients are biased and inconsistent; predictions may be systematically off.

**Example:** If house prices have increasing variance with income (heteroscedasticity), OLS standard errors are wrong. You might conclude income is significant when it's not, or vice versa.

**Follow-up:** "What remedies would you apply for each violated assumption?"


## Conclusion

This notebook demonstrated a complete linear regression workflow:

1. **Loaded** the California Housing dataset (20,640 rows, 10 columns)
2. **Preprocessed** data: imputed missing values, one-hot encoded categoricals
3. **Split** data into 80/20 train/test sets
4. **Built** simple linear regression using `median_income`
5. **Built** multiple linear regression using all features
6. **Evaluated** both models with MAE, MSE, and R²
7. **Visualized** regression lines, actual vs predicted, and residuals
8. **Interpreted** coefficients and checked multicollinearity with correlation and VIF
9. **Analyzed** residuals to validate assumptions
10. **Prepared** for 8 common regression interview questions

### Key Takeaways
- Always split data before preprocessing to avoid leakage
- Simple LR builds intuition; multiple LR captures more complexity
- R² quantifies explained variance but doesn't tell the whole story
- Multicollinearity doesn't hurt predictions much but makes coefficients unreliable
- Residual analysis is essential for validating model assumptions
- Always interpret coefficients in the context of feature scales
- Linear regression is the foundation for understanding regularization and advanced models

---

## How to Use This Notebook

1. Ensure the virtual environment is activated: `.env\Scripts\Activate.ps1`
2. Launch Jupyter: `jupyter notebook`
3. Open `notebooks/linear_regression.ipynb`
4. Select kernel: **Python (venv-elevate)**
5. Run cells sequentially from top to bottom
6. Review each output and interpretation
7. Use the interview Q&A to prepare for technical discussions
